In [ ]:
import numpy as np
import pandas as pd

# Kept as-is: the two data loaders. NOTHING is imported from gbp.consumers.simulator.
from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation

In [2]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="../data/raw/202602-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1))

In [3]:
periods_df = graph_data.periods_df
initial_inventory_df = graph_data.initial_inventory_df
potential_trips_df = graph_data.potential_trips_df
historical_flows_df = graph_data.historical_flows_df

In [4]:
# The flow journal is the single source of truth for what happened in a run.
FLOW_EVENT_COLUMNS = [
    "event_id", "period_id", "flow_id", "flow_type", "event_type", "commodity_category",
    "source_id", "planned_target_id", "realized_target_id",
    "start_period", "planned_end_period", "realized_end_period",
    "resource_id", "quantity", "reason",
]
FLOW_EVENT_DTYPES = {
    "period_id": "Int64", "flow_id": "string", "flow_type": "string", "event_type": "string",
    "commodity_category": "string", "source_id": "string", "planned_target_id": "string",
    "realized_target_id": "string", "start_period": "Int64", "planned_end_period": "Int64",
    "realized_end_period": "Int64", "resource_id": "string", "quantity": "Int64", "reason": "string",
}
print(len(FLOW_EVENT_COLUMNS), "columns;", len(FLOW_EVENT_DTYPES), "typed")

15 columns; 14 typed


In [5]:
def _typed_events(events_df):
    """Cast event columns to the canonical dtypes so frames concat cleanly."""
    for col, dtype in FLOW_EVENT_DTYPES.items():
        events_df[col] = events_df[col].astype(dtype)
    events_df["event_order"] = events_df["event_order"].astype("int64")
    return events_df


def departed_events(trips):
    """One ``departed`` event row per trip leaving this period (event_order=0)."""
    return _typed_events(pd.DataFrame({
        "period_id":           trips["start_period"],
        "flow_id":             trips["flow_id"],
        "flow_type":           "user_trip",
        "event_type":          "departed",
        "commodity_category":  trips["commodity_category"],
        "source_id":           trips["source_id"],
        "planned_target_id":   trips["planned_target_id"],
        "realized_target_id":  pd.NA,
        "start_period":        trips["start_period"],
        "planned_end_period":  trips["planned_end_period"],
        "realized_end_period": pd.NA,
        "resource_id":         pd.NA,
        "quantity":            1,
        "reason":              pd.NA,
        "event_order":         0,
    }))


def arrived_events(in_transit_due, period_id):
    """One ``arrived`` event row per docking flow (event_order=1).

    ``period_id`` is the docking period: a scalar int when a whole batch docks
    together (the simulator), or a per-row Series (the historical log builder).
    """
    return _typed_events(pd.DataFrame({
        "period_id":           period_id,
        "flow_id":             in_transit_due["flow_id"],
        "flow_type":           "user_trip",
        "event_type":          "arrived",
        "commodity_category":  in_transit_due["commodity_category"],
        "source_id":           in_transit_due["source_id"],
        "planned_target_id":   in_transit_due["planned_target_id"],
        "realized_target_id":  in_transit_due["planned_target_id"],
        "start_period":        in_transit_due["start_period"],
        "planned_end_period":  in_transit_due["planned_end_period"],
        "realized_end_period": period_id,
        "resource_id":         pd.NA,
        "quantity":            1,
        "reason":              pd.NA,
        "event_order":         1,
    }))

In [6]:
# Demo: turn 2 toy trips into their departed + arrived event rows.
demo_trips = pd.DataFrame({
    "flow_id":            pd.array(["t0", "t1"], dtype="string"),
    "source_id":          pd.array(["S1", "S2"], dtype="string"),
    "planned_target_id":  pd.array(["S2", "S3"], dtype="string"),
    "commodity_category": pd.array(["classic_bike", "electric_bike"], dtype="string"),
    "start_period":       pd.array([0, 0], dtype="Int64"),
    "planned_end_period": pd.array([0, 1], dtype="Int64"),
})

In [7]:
def empty_in_transit():
    """Empty in-transit table (a ``departed``-event frame with no rows)."""
    return departed_events(pd.DataFrame({
        "flow_id":            pd.Series(dtype="string"),
        "source_id":          pd.Series(dtype="string"),
        "planned_target_id":  pd.Series(dtype="string"),
        "commodity_category": pd.Series(dtype="string"),
        "start_period":       pd.Series(dtype="Int64"),
        "planned_end_period": pd.Series(dtype="Int64"),
    }))


def empty_flows_journal():
    """Empty append-only flow journal (no ``event_id`` until finalize)."""
    journal = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in FLOW_EVENT_DTYPES.items()})
    journal["event_order"] = pd.Series(dtype="int64")
    return journal

In [8]:
def adjust_inventory(inventory, deltas):
    """Add signed ``delta`` per (facility_id, commodity_category)."""
    out = inventory.merge(deltas, on=["facility_id", "commodity_category"], how="outer")
    out["quantity"] = out["quantity"].fillna(0) + out["delta"].fillna(0)
    return out[["facility_id", "commodity_category", "quantity"]]


def departure_deltas(trips):
    """-1 per departing bike, grouped by (source, commodity)."""
    deltas = (
        trips.groupby(["source_id", "commodity_category"]).size()
        .reset_index(name="delta").rename(columns={"source_id": "facility_id"})
    )
    deltas["delta"] = -deltas["delta"]
    return deltas


def arrival_deltas(in_transit_due):
    """+1 per docking bike, grouped by (target, commodity)."""
    return (
        in_transit_due.groupby(["planned_target_id", "commodity_category"]).size()
        .reset_index(name="delta").rename(columns={"planned_target_id": "facility_id"})
    )

In [ ]:
# FormDepartures helpers: gate aggregated demand by the stock on hand, then draw
# concrete trips for the part that actually departs. This is where a stockout bites
# (dormant in an exact replay, where stock always covers the historical demand).
def realize_demand(demand_now, inventory):
    """Realized departures per (source, commodity): ``min(demand, available stock)``.

    Demand above the stock on hand is lost to a stockout. Stock is per commodity,
    so classic and electric demand are gated independently.

    Returns
    -------
    pandas.DataFrame
        ``start_period``, ``source_id``, ``commodity_category``, ``realized``, ``lost``.
    """
    stock = inventory.rename(columns={"facility_id": "source_id", "quantity": "available"})
    out = demand_now.merge(stock, on=["source_id", "commodity_category"], how="left")
    out["available"] = out["available"].fillna(0)
    out["realized"] = out[["quantity", "available"]].min(axis=1).astype("int64")
    out["lost"] = (out["quantity"] - out["realized"]).astype("int64")
    return out[["start_period", "source_id", "commodity_category", "realized", "lost"]]


def departure_deltas_from_counts(realized):
    """``-realized`` per (source, commodity) for the inventory decrement."""
    d = (realized[realized["realized"] > 0]
         .rename(columns={"source_id": "facility_id", "realized": "delta"})
         [["facility_id", "commodity_category", "delta"]].copy())
    d["delta"] = -d["delta"]
    return d


# FormPotentialTrips helpers: the OD demand model. Instead of replaying the exact
# historical trip rows, departures are split across destinations by the OD matrix
# P(target | source, commodity), and each OD pair's mean historical duration sets
# the arrival period. The two functions below build that matrix and apply it.
def build_od_matrix(potential_trips):
    """Origin-destination demand model from concrete historical trips.

    For each ``(source_id, planned_target_id, commodity_category)``:

    - ``count`` -- number of historical trips on the pair,
    - ``probability`` -- ``P(target | source, commodity)``, normalized within
      each ``(source, commodity)``,
    - ``duration`` -- mean trip length in periods (``planned_end - start``),
      rounded to whole periods.

    This is the historical OD matrix; in the trivial case the simulated and state
    OD matrices are equal to it.
    """
    trips = potential_trips.copy()
    trips["duration"] = trips["planned_end_period"] - trips["start_period"]
    od = (
        trips.groupby(["source_id", "planned_target_id", "commodity_category"], as_index=False)
        .agg(count=("flow_id", "size"), duration=("duration", "mean"))
    )
    totals = od.groupby(["source_id", "commodity_category"])["count"].transform("sum")
    od["probability"] = od["count"] / totals
    od["duration"] = od["duration"].round().astype("Int64")
    return od


def form_potential_trips(departures, od_matrix, period_id):
    """Split each source's departures across targets by the OD probabilities.

    Each ``(source, commodity)`` releases ``quantity`` bikes this period; the OD
    matrix ``P(target | source, commodity)`` decides their destinations. The
    expected count per target (``departures * probability``) is rounded to whole
    bikes by the largest-remainder method, so the per-source total is preserved
    exactly. Each OD pair's mean historical duration sets the arrival period.

    Parameters
    ----------
    departures : pandas.DataFrame
        Realized departures this period: ``source_id``, ``commodity_category``,
        ``quantity``.
    od_matrix : pandas.DataFrame
        OD demand model from :func:`build_od_matrix`.
    period_id : int
        The current (departure) period.

    Returns
    -------
    pandas.DataFrame
        ``period_id``, ``source_id``, ``target_id``, ``commodity_category``,
        ``quantity``, ``planned_end_period`` -- only rows with ``quantity > 0``.
    """
    cols = ["period_id", "source_id", "target_id", "commodity_category",
            "quantity", "planned_end_period"]
    dep = departures[departures["quantity"] > 0]
    if dep.empty:
        return pd.DataFrame({c: pd.Series(dtype="object") for c in cols})

    m = dep.merge(od_matrix, on=["source_id", "commodity_category"], how="left")
    m = m[m["probability"].notna()].copy()
    m["expected"] = m["quantity"] * m["probability"]
    m["base"] = np.floor(m["expected"]).astype("int64")
    m["remainder"] = m["expected"] - m["base"]

    # Largest-remainder rounding: hand the per-source shortfall to the targets
    # with the largest fractional parts, so sum(quantity) == departures exactly.
    m = m.sort_values(["source_id", "commodity_category", "remainder"],
                      ascending=[True, True, False])
    grp = m.groupby(["source_id", "commodity_category"])
    m["rank"] = grp.cumcount()
    m["shortfall"] = m["quantity"] - grp["base"].transform("sum")
    m["qty"] = m["base"] + (m["rank"] < m["shortfall"]).astype("int64")
    m = m[m["qty"] > 0]

    return pd.DataFrame({
        "period_id":          period_id,
        "source_id":          m["source_id"].values,
        "target_id":          m["planned_target_id"].values,
        "commodity_category": m["commodity_category"].values,
        "quantity":           m["qty"].astype("Int64").values,
        "planned_end_period": (period_id + m["duration"]).astype("Int64").values,
    })


def expand_potential_trips(potential_trips, period_id):
    """Expand aggregate OD potential trips into one concrete departed row per bike.

    Each aggregate row carries ``quantity`` identical bikes; this repeats it into
    that many trip rows and assigns a simulator ``flow_id`` (``sim_`` prefix so it
    cannot collide with the historical ``hist_`` ids).
    """
    rep = (potential_trips.loc[potential_trips.index.repeat(potential_trips["quantity"])]
           .reset_index(drop=True))
    return pd.DataFrame({
        "flow_id":            "sim_" + str(period_id) + "_" + rep.index.astype("string"),
        "source_id":          rep["source_id"],
        "planned_target_id":  rep["target_id"],
        "commodity_category": rep["commodity_category"],
        "start_period":       period_id,
        "planned_end_period": rep["planned_end_period"],
    })

In [10]:
# Capacity-aware docking and overflow redirect: the part that bites above the
# historical baseline (dormant in an exact replay, where capacity never binds).
def free_docks(inventory, capacities):
    """Free dock slots per facility: capacity minus bikes currently docked.

    Classic and electric bikes share the same physical docks, so occupancy is the
    total stock across commodities.

    Returns
    -------
    pandas.Series
        ``facility_id -> free slots`` (clipped at zero).
    """
    occupied = inventory.groupby("facility_id")["quantity"].sum()
    capacity = capacities.set_index("facility_id")["capacity"]
    idx = capacity.index.union(occupied.index)
    free = capacity.reindex(idx).fillna(0) - occupied.reindex(idx).fillna(0)
    return free.clip(lower=0).astype("int64")


def dock_up_to_capacity(due, free):
    """Split docking flows at their planned target into ``(fits, overflow)``.

    Within each target the first ``free`` flows (in row order) dock; the rest are
    overflow. Vectorized through a per-target cumulative count -- no Python loop.
    """
    if due.empty:
        return due, due
    rank = due.groupby("planned_target_id").cumcount()
    capacity_here = due["planned_target_id"].map(free).fillna(0)
    fits = rank < capacity_here
    return due[fits], due[~fits]


def redirected_events(flows, realized_target_id, period_id):
    """One ``redirected`` docking event per overflow flow that docked elsewhere.

    ``realized_target_id`` is the station actually docked at (a per-row Series),
    which differs from ``planned_target_id``; ``reason`` records why.
    """
    return _typed_events(pd.DataFrame({
        "period_id":           period_id,
        "flow_id":             flows["flow_id"],
        "flow_type":           "user_trip",
        "event_type":          "redirected",
        "commodity_category":  flows["commodity_category"],
        "source_id":           flows["source_id"],
        "planned_target_id":   flows["planned_target_id"],
        "realized_target_id":  realized_target_id,
        "start_period":        flows["start_period"],
        "planned_end_period":  flows["planned_end_period"],
        "realized_end_period": period_id,
        "resource_id":         pd.NA,
        "quantity":            1,
        "reason":              "dock_full",
        "event_order":         1,
    }))


def _nearest_free_station(targets, free, geo):
    """Nearest *other* station with a free dock for each station id in ``targets``.

    Distance is squared Euclidean on (lat, lng) -- enough to rank neighbours at
    this prototype stage. Returns a Series aligned to ``targets`` (NA if none).
    """
    candidates = free[free > 0].index
    coords = geo.set_index("facility_id")[["lat", "lng"]]
    origins = (coords.loc[coords.index.intersection(targets.unique())]
               .reset_index().rename(columns={"facility_id": "origin"}))
    cand = (coords.loc[coords.index.intersection(candidates)]
            .reset_index().rename(columns={"facility_id": "candidate"}))
    pairs = origins.merge(cand, how="cross")
    pairs = pairs[pairs["origin"] != pairs["candidate"]]
    pairs["dist2"] = (pairs["lat_x"] - pairs["lat_y"]) ** 2 + (pairs["lng_x"] - pairs["lng_y"]) ** 2
    nearest = pairs.sort_values("dist2").drop_duplicates("origin").set_index("origin")["candidate"]
    return targets.map(nearest)


def redirect_overflow(inventory, capacities, geo, overflow, period_id):
    """Greedily dock overflow flows at the nearest station with a free dock.

    Rounds, not per-row loops: each round maps every still-unplaced flow to its
    nearest free station, docks up to capacity there, applies the arrivals to
    inventory, and repeats with the leftovers until none remain or no dock is free
    anywhere. The within-batch capacity coupling is therefore honoured exactly.

    Returns
    -------
    tuple of (pandas.DataFrame, pandas.DataFrame, pandas.DataFrame)
        The ``redirected`` events, the inventory with the redirected arrivals
        applied, and any flows that found no free dock anywhere (lost).
    """
    events = []
    remaining = overflow
    while not remaining.empty:
        free = free_docks(inventory, capacities)
        if not (free > 0).any():
            break
        target = _nearest_free_station(remaining["planned_target_id"], free, geo)
        placed = remaining.assign(realized_target_id=target)
        placed = placed[placed["realized_target_id"].notna()]
        if placed.empty:
            break
        rank = placed.groupby("realized_target_id").cumcount()
        fits = rank < placed["realized_target_id"].map(free)
        docked = placed[fits]
        events.append(redirected_events(docked, docked["realized_target_id"], period_id))
        deltas = (docked.groupby(["realized_target_id", "commodity_category"]).size()
                  .reset_index(name="delta").rename(columns={"realized_target_id": "facility_id"}))
        inventory = adjust_inventory(inventory, deltas)
        remaining = placed[~fits].drop(columns="realized_target_id")
    events_df = (pd.concat(events, ignore_index=True) if events
                 else redirected_events(overflow.iloc[:0], overflow["planned_target_id"].iloc[:0], period_id))
    return events_df, inventory, remaining

In [11]:
# Demo: how a departure changes inventory.
demo_inv = pd.DataFrame({
    "facility_id":        pd.array(["S1", "S2", "S3"], dtype="string"),
    "commodity_category": pd.array(["classic_bike", "electric_bike", "classic_bike"], dtype="string"),
    "quantity":           [5, 2, 0],
})

In [12]:
def finalize_flows(journal):
    """Order the accumulated journal and assign a monotonic ``event_id``.

    Sort keys: period_id, then flow_id, then event_order (departed=0 before
    arrived=1). Both the historical log and a replay journal go through this one
    function, so they share ordering and event_id assignment by construction.
    """
    flows = journal.copy()
    for col, dtype in FLOW_EVENT_DTYPES.items():
        flows[col] = flows[col].astype(dtype)
    flows = flows.sort_values(["period_id", "flow_id", "event_order"]).reset_index(drop=True)
    flows["event_id"] = flows.index.astype("Int64")
    return flows[FLOW_EVENT_COLUMNS]

In [13]:
# Demo: an unordered 4-row journal -> finalized (sorted, event_id assigned).
shuffled = pd.concat([
    arrived_events(demo_trips, demo_trips["planned_end_period"]),
    departed_events(demo_trips),
], ignore_index=True)

In [ ]:
periods_df_demo = pd.DataFrame({
    "period_id":       [0, 1, 2],
    "start_timestamp": pd.to_datetime(["2026-02-01 00:00", "2026-02-01 01:00", "2026-02-01 02:00"]),
    "end_timestamp":   pd.to_datetime(["2026-02-01 01:00", "2026-02-01 02:00", "2026-02-01 03:00"]),
})

historical_potential_trips_df_demo = pd.DataFrame({
    "flow_id":            pd.array(["A", "C", "E", "B", "F"], dtype="string"),
    "source_id":          pd.array(["S1", "S2", "S1", "S1", "S1"], dtype="string"),
    "planned_target_id":  pd.array(["S2", "S1", "S2", "S3", "S3"], dtype="string"),
    "commodity_category": pd.array(["classic_bike"] * 5, dtype="string"),
    "start_period":       pd.array([0, 0, 0, 0, 0], dtype="Int64"),
    "planned_end_period": pd.array([0, 0, 0, 1, 1], dtype="Int64"),
})

historical_demand_df_demo = (
    historical_potential_trips_df_demo
    .groupby(["start_period", "source_id", "commodity_category"], as_index=False)
    .size().rename(columns={"size": "quantity"})
)

# OD demand model with probabilities P(target | source, commodity) and mean
# durations. Trivial case: the simulated and state OD matrices equal the
# historical one (no demand shift), so a base run reproduces the historical OD
# structure rather than each concrete trip row.
historical_od_matrix_df_demo = build_od_matrix(historical_potential_trips_df_demo)
simulated_od_matrix_df_demo = historical_od_matrix_df_demo
state_od_matrix_df = simulated_od_matrix_df_demo

initial_inventory_df_demo = pd.DataFrame({
    "facility_id":        pd.array(["S1", "S1", "S2", "S2", "S3", "S3"], dtype="string"),
    "commodity_category": pd.array(["classic_bike", "electric_bike", "classic_bike", "electric_bike", "classic_bike", "electric_bike"], dtype="string"),
    "quantity":           [5, 2, 5, 2, 0, 0],
})
capacities_df_demo = pd.DataFrame({
    "facility_id": pd.array(["S1", "S2", "S3"], dtype="string"),
    "capacity":    [10, 7, 1],
})
facilities_geo_df_demo = pd.DataFrame({
    "facility_id": pd.array(["S1", "S2", "S3"], dtype="string"),
    "lat":         [0.0, 0.0, 5.0],
    "lng":         [0.0, 1.0, 5.0],
})

In [15]:
# The working state: three plain variables (the engine's SimulationState, unpacked).
inventory_df  = initial_inventory_df_demo.copy()
in_transit    = empty_in_transit()
flows_journal = empty_flows_journal()

In [ ]:
inventory_df  = initial_inventory_df_demo.copy()
in_transit    = empty_in_transit()
flows_journal = empty_flows_journal()

state_demand_df = historical_demand_df_demo

for t in periods_df_demo["period_id"]:
    # Phase 1 - ArrivalsPreviousPhase: dock earlier departures arriving now, up to free docks.
    due_prev = in_transit[(in_transit["planned_end_period"] == t) & (in_transit["start_period"] < t)]
    docked_prev, overflow_prev = dock_up_to_capacity(due_prev, free_docks(inventory_df, capacities_df_demo))
    if not docked_prev.empty:
        flows_journal = pd.concat([flows_journal, arrived_events(docked_prev, t)], ignore_index=True)
        inventory_df  = adjust_inventory(inventory_df, arrival_deltas(docked_prev))
    in_transit = in_transit.drop(due_prev.index)

    # Phase 2 - OverflowRedirectPreviousPhase
    if not overflow_prev.empty:
        ev_red, inventory_df, lost = redirect_overflow(
            inventory_df, capacities_df_demo, facilities_geo_df_demo, overflow_prev, t)
        flows_journal = pd.concat([flows_journal, ev_red], ignore_index=True)
        if not lost.empty:
            print(f"period {t}: {len(lost)} flow(s) found no free dock (lost)")

    # Phase 3 - FormDepartures: how many bikes depart per (source, commodity), gated by stock.
    demand_now   = state_demand_df[state_demand_df["start_period"] == t]
    realized_dep = realize_demand(demand_now, inventory_df)
    inventory_df = adjust_inventory(inventory_df, departure_deltas_from_counts(realized_dep))
    lost_now     = int(realized_dep["lost"].sum())
    if lost_now:
        print(f"period {t}: {lost_now} trip(s) lost to stockout")

    # Phase 4 - FormPotentialTrips: split the realized departures across targets via
    # the OD matrix P(target | source, commodity), then expand to concrete trips.
    dep_counts    = realized_dep.rename(columns={"realized": "quantity"})
    potential_now = form_potential_trips(dep_counts, state_od_matrix_df, t)
    trips_now     = expand_potential_trips(potential_now, t)
    if not trips_now.empty:
        ev_dep        = departed_events(trips_now)
        in_transit    = pd.concat([in_transit, ev_dep], ignore_index=True)
        flows_journal = pd.concat([flows_journal, ev_dep], ignore_index=True)

    # Phase 5 - ArrivalsPhase: dock bikes that depart and arrive within this same period, up to free docks.
    due_same = in_transit[(in_transit["planned_end_period"] == t) & (in_transit["start_period"] == t)]
    docked_same, overflow_same = dock_up_to_capacity(due_same, free_docks(inventory_df, capacities_df_demo))
    if not docked_same.empty:
        flows_journal = pd.concat([flows_journal, arrived_events(docked_same, t)], ignore_index=True)
        inventory_df  = adjust_inventory(inventory_df, arrival_deltas(docked_same))
    in_transit = in_transit.drop(due_same.index)

    # Phase 6 - OverflowRedirect: same redirect rule for same-period arrivals.
    if not overflow_same.empty:
        ev_red, inventory_df, lost = redirect_overflow(
            inventory_df, capacities_df_demo, facilities_geo_df_demo, overflow_same, t)
        flows_journal = pd.concat([flows_journal, ev_red], ignore_index=True)
        if not lost.empty:
            print(f"period {t}: {len(lost)} flow(s) found no free dock (lost)")

print("inventory after the demo run:")
display(inventory_df.sort_values(["facility_id", "commodity_category"]).reset_index(drop=True))
print("flows journal (planned vs realized target -- a redirect is where they differ):")
display(flows_journal[["period_id", "event_type", "flow_id", "planned_target_id", "realized_target_id", "reason"]]
        .sort_values(["period_id", "flow_id"]).reset_index(drop=True))